# Practice 2 · Wide vs Deep Networks

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |

### I. Background

Two networks can reach a similar loss with very different numbers of parameters. A **wide** network
adds units to one hidden layer; a **deep** network adds layers. In this practice you count the
parameters of both and train them on the same data, then explain what you see.

### II. Task Requirements

1. Set `SID`; the training subset depends on it.
2. Load a subset of CIFAR10 and print the shape of one batch.
3. Count the parameters of the given wide network and deep network.
4. Train both for the same number of epochs and record the last loss; save the two loss curves as
   `wide_vs_deep_<SID>.png`.

Steps: Open the notebook in Haima, fill in the missing code, and run it.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| `n_sub` | size of the training subset | 3000 |
| `xb`, `yb` | one batch | `(B, 3, 32, 32)` and `(B,)` |
| `n_wide`, `n_deep` | number of parameters | 3157002 / 830250 |
| `loss_wide`, `loss_deep` | last-epoch training loss | a few tenths, smaller than 2.3 |
| `wide_vs_deep_<SID>.png` | the two loss curves | one file |

## Question 1 · Load a subset of CIFAR10

CIFAR10 is already on the platform. Keep the subset size and the batch size as they are, and write
the three missing lines.

In [2]:
import os
import torch, torch.nn as nn
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

# 🟢 EDIT: set SID to the last two digits of your student ID
SID = 23
torch.manual_seed(1000 + SID)

# 🟡 READ: the subset and the batch size are fixed, so everyone trains on the same amount of data
N_SUB, BATCH = 3000, 512

transform = transforms.Compose([transforms.ToTensor()])
# 🟡 READ: CIFAR10 ships with the platform; the check below only avoids an accidental 170 MB download
DATA_ROOT = "data"
_cached = os.path.exists(os.path.join(DATA_ROOT, "cifar-10-batches-py"))
if not _cached:
    print("CIFAR10 not found under ./data - it will be downloaded (about 170 MB), please wait ...")
full = CIFAR10(DATA_ROOT, train=True, download=not _cached, transform=transform)

# 🟢 EDIT: build the subset and the DataLoader
g = torch.Generator().manual_seed(1000 + SID)
idx = torch.randperm(len(full), generator=g)[:N_SUB].tolist()
ds = Subset(full, idx)
# YOUR CODE HERE
dl = DataLoader(ds, batch_size=BATCH, shuffle=True)

xb, yb = next(iter(dl))
n_sub = len(ds)
print("subset =", n_sub, " one batch =", tuple(xb.shape), tuple(yb.shape))

subset = 3000  one batch = (512, 3, 32, 32) (512,)


## Question 2 · Count the parameters

The two networks are given below. Count their parameters with
`sum(p.numel() for p in net.parameters())` and fill in the two numbers.

In [3]:
# 🟡 READ: the wide network (one hidden layer) and the deep network (five hidden layers)
def make_wide():
    return nn.Sequential(nn.Flatten(), nn.Linear(3072, 1024), nn.ReLU(), nn.Linear(1024, 10))

def make_deep():
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(3072, 256), nn.ReLU(),
                         nn.Linear(256, 128), nn.ReLU(),
                         nn.Linear(128, 64), nn.ReLU(),
                         nn.Linear(64, 32), nn.ReLU(),
                         nn.Linear(32, 10))

wide, deep = make_wide(), make_deep()

# 🟢 EDIT: compute the number of parameters of both networks
#   net.parameters() lists every weight and bias tensor of a network
#   p.numel() tells how many values one tensor holds
#   Add them up for wide and for deep, and store the two sums in n_wide and n_deep
# YOUR CODE HERE
n_wide = sum(p.numel() for p in wide.parameters())
n_deep = sum(p.numel() for p in deep.parameters())

print("wide net: %d parameters" % n_wide)
print("deep net: %d parameters" % n_deep)
print("ratio    : %.2f" % (n_wide / n_deep))

wide net: 3157002 parameters
deep net: 830250 parameters
ratio    : 3.80


## Question 3 · Train both and compare the curves

Two epochs each are enough for the comparison. Fill in the training loop and save the figure.

In [4]:
import time, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]

EPOCHS = 3

def train(net):
    net.train()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss()
    curve = []
    for ep in range(EPOCHS):
        tot, nb = 0.0, 0
        for bx, by in dl:
            # 🟢 EDIT: one training step (three lines + backward)
            #   (1) clear the gradients left from the previous batch,
            #   (2) run the current batch through the network and compute the loss,
            #   (3) run backpropagation to get the gradients,
            #   (4) update the parameters with the optimizer.
            #   The objects you need are: opt, lossf, net, bx, by.
            # YOUR CODE HERE
            opt.zero_grad()
            loss = lossf(net(bx), by)
            loss.backward()
            opt.step()
            tot += float(loss) * len(by); nb += len(by)
        curve.append(tot / nb)
        print("  epoch %d  loss %.4f" % (ep + 1, curve[-1]))
    return curve

print("wide:");  cw = train(make_wide())
print("deep:");  cd = train(make_deep())
loss_wide, loss_deep = cw[-1], cd[-1]

plt.figure(figsize=(6, 3.4))
plt.plot(range(1, EPOCHS + 1), cw, "o-", color="#1F4E79", label="wide (%d params)" % n_wide)
plt.plot(range(1, EPOCHS + 1), cd, "s-", color="#2E8B57", label="deep (%d params)" % n_deep)
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title("Wide vs deep - SID %d" % SID)
plt.tight_layout()
plt.savefig("wide_vs_deep_%d.png" % SID, dpi=110)
plt.show()
print("last losses: wide %.4f  deep %.4f" % (loss_wide, loss_deep))
# 🟢 EDIT: answer the two questions about what you just measured.
# YOUR CODE HERE
print("B")
print("B")


wide:


C:\Users\Administrator\AppData\Local\Temp\ipykernel_11836\521366199.py:27: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  tot += float(loss) * len(by); nb += len(by)


  epoch 1  loss 2.8843
  epoch 2  loss 2.2283
  epoch 3  loss 2.0588
deep:
  epoch 1  loss 2.2962
  epoch 2  loss 2.2562
  epoch 3  loss 2.1927
last losses: wide 2.0588  deep 2.1927
B
B


C:\Users\Administrator\AppData\Local\Temp\ipykernel_11836\521366199.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Question 4 · Which network uses fewer parameters?

- (A) the wide one      (B) the deep one      (C) they are equal


B. The deep network uses 830,250 parameters, compared with 3,157,002 for the wide network.

## Question 5 · Why can the deep network reach a similar loss with fewer parameters?

- (A) its capacity is smaller, so it cannot overfit
- (B) it combines features layer by layer, which represents the function more efficiently
- (C) Adam only works for deep networks


B. The deep network combines features layer by layer, which can represent complex functions more efficiently with fewer parameters. Similar training loss must still be confirmed by the measured results.